In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
%matplotlib inline
# retina
%config InlineBackend.figure_format = 'retina'

In [ ]:
from sklearn.datasets import make_blobs

In [ ]:
X, y = make_blobs(centers=2, n_samples=100, random_state=0, cluster_std=1.5)

In [ ]:
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=cm.autumn)
plt.gca().set_aspect('equal')

In [ ]:
from sklearn import svm
X_train = X
y_train = y
kernel = 'linear'
for fig_num, C in enumerate([0.0001, 0.001, 0.1, 1, 100, 10000][:]):
    clf = svm.SVC(kernel=kernel,C = C )
    clf.fit(X, y)

    plt.figure(fig_num)
    plt.clf()
    plt.scatter(X[:, 0], X[:, 1], c=y, zorder=10, cmap=plt.cm.autumn,
                edgecolor='k',s=80)


    plt.axis('tight')
    x_min = X[:, 0].min()-1
    x_max = X[:, 0].max()+1
    y_min = X[:, 1].min()-1
    y_max = X[:, 1].max()+1

    XX, YY = np.mgrid[x_min:x_max:200j, y_min:y_max:200j]
    Z = clf.decision_function(np.c_[XX.ravel(), YY.ravel()])
   
    # Put the result into a color plot
    Z = Z.reshape(XX.shape)
    #plt.pcolormesh(XX, YY, Z > 0, cmap=plt.cm.autumn, alpha=0.2)

    plt.contour(XX, YY, Z, colors=['k', 'k', 'k'],
                linestyles=['--', '-', '--'], levels=[-1, 0, 1])
    plt.scatter(clf.support_vectors_[:, 0],
                clf.support_vectors_[:, 1],
                s=5, lw=0.001, facecolors='none',zorder=10,c='k', edgecolors='none');


    plt.title("Linear Kernel with C= {}\n Number of support vectors = {}".format(C, len(clf.support_vectors_)))


In [ ]:
### from scratch implementation in Torch

In [ ]:
import torch.nn as nn
import torch
import torch.optim as optim

class OurSVM(nn.Module):
    def __init__(self, C = 1.0):
        super(OurSVM, self).__init__()
        self.C = C
                
    def forward(self, x):
        return torch.matmul(x, self.w) + self.b

    def hinge_loss(self, y_pred, y_true):
        return torch.mean(torch.max(torch.zeros_like(y_pred), 1 - y_true * y_pred)) + (torch.sum(self.w**2)) / (2 * self.C)

    def fit(self, X, y, epochs=100, lr=3e-4):
        self.w = nn.Parameter(torch.randn(X.shape[1], 1, requires_grad=True))
        self.b = nn.Parameter(torch.randn(1, requires_grad=True))
        optimizer = optim.Adam(self.parameters(), lr=lr)
        for epoch in range(epochs):
            optimizer.zero_grad()
            y_pred = self.forward(X)
            loss = self.hinge_loss(y_pred, y)
            print(f'Epoch {epoch}, Loss: {loss.item()}')
            loss.backward()
            optimizer.step()

In [ ]:
our_svm = OurSVM(C=1.0)
X_train = torch.tensor(X, dtype=torch.float32)
y_train = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)
y_train[y_train == 0] = -1
our_svm.fit(X_train, y_train, epochs=4500, lr=3e-3)

In [ ]:
our_svm.w, our_svm.b

In [ ]:
sklearn_svm = svm.SVC(kernel='linear', C=1)
sklearn_svm.fit(X, y)
sklearn_svm.coef_, sklearn_svm.intercept_